# Colab starter

Upload `build/kaggle-starter-colab.zip` to `/content` and put competition data on Drive. Prepare `train.csv` and `valid.csv` for the example.


In [ ]:
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
with tempfile.TemporaryDirectory() as directory:
    wheelhouse = Path(directory)
    with zipfile.ZipFile("/content/kaggle-starter-colab.zip") as archive:
        archive.extractall(wheelhouse)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--find-links", str(wheelhouse),
        "-r", str(wheelhouse / "requirements.txt"),
    ], check=True)
    wheel = next(wheelhouse.glob("kaggle_starter-*.whl"))
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheel),
    ], check=True)


In [ ]:
import os

from kaggle_starter.config import LoaderConfig, RuntimeConfig, TrainConfig, get_config
from kaggle_starter.models.baseline.config import BaselineExperimentConfig, ModelConfig
from kaggle_starter.runtime.environment import apply_environment_defaults, get_secret

apply_environment_defaults()
os.environ["KS_DATA__COMPETITION_DIR"] = '/content/drive/MyDrive/data/competition'

RUN_NAME = "baseline"
DATASET_ID = "your-data-v1"
WANDB_OFFLINE = False  # True for submissions
os.environ["KS_TRACKING__OFFLINE"] = str(WANDB_OFFLINE).lower()
if not WANDB_OFFLINE and (api_key := get_secret("WANDB_API_KEY")):
    os.environ["KS_TRACKING__API_KEY"] = api_key

config = get_config(
    BaselineExperimentConfig,
    model=ModelConfig(feature_columns=["feature"], target_column="target"),
    runtime=RuntimeConfig(accelerator="auto", devices="auto"),
    loader=LoaderConfig(batch_size=128, val_batch_size=64, num_workers=0),
    train=TrainConfig(max_epochs=5),
)


In [ ]:
import json
import subprocess
import sys

if config.tracking.api_key is not None:
    os.environ["KS_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config.data.output_dir.mkdir(parents=True, exist_ok=True)
config_path = config.data.output_dir / f"{RUN_NAME}-config.json"
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")

subprocess.run([
    sys.executable, "-m", "kaggle_starter.models.baseline.fit",
    "--config", str(config_path), "--name", RUN_NAME, "--dataset-id", DATASET_ID,
    "--train", str(config.data.competition_dir / "train.csv"),
    "--valid", str(config.data.competition_dir / "valid.csv"),
], check=True)
